# 17.5 Практическая работа

## Интеграция данных из разных источников

Цель работы - сформировать единую витрину данных по действующим страховым полисам.

В итоговую витрину необходимо добавить:
- данные страхового полиса;
- инициалы страхователя;
- пол;
- возраст;
- информацию об убытках;
- сумму выплаты по убытку;
- денежные показатели, пересчитанные в доллары США по актуальному курсу валют.

Исходные данные представлены в трёх Excel-файлах и данных о курсах валют, получаемых через API.

In [1]:
import pandas as pd
import requests

clients = pd.read_excel("CASE_CLIENTS.xlsx")
contracts = pd.read_excel("CASE_CONTRACTS.xlsx")
losses = pd.read_excel("CASE_LOSSES.xlsx")

## 1. Загрузка и первичная проверка данных

Загрузим три исходных Excel-файла и проверим их размер, структуру и названия столбцов.

In [2]:
print("Клиенты:", clients.shape)
print("Договоры:", contracts.shape)
print("Убытки:", losses.shape)

display(clients.head())
display(contracts.head())
display(losses.head())

Клиенты: (3711, 6)
Договоры: (3711, 10)
Убытки: (45, 4)


,client_id,last_name,first_name,middle_name,age,sex
0,10000041307,З*****,Р*****,О*****,30,F
1,10000133158,И*****,М*****,Ю*****,37,M
2,100003773,Е*****,О*****,С*****,50,M
3,10000535718,Е*****,Ф*****,Ч*****,45,M
4,10000583119,И*****,Ф*****,Ф*****,28,M


,contract_id,contract_num,product_name,client_id,contract_status,currency_name,duration,country,price,insurance_amount
0,21111219,ТТЕ7227715*****,Страхование путешественников,10161870404,Действует,Российский рубль,10,Беларусь,1096,2000000
1,21111381,ТТЕ7227715*****,Страхование путешественников,1017159879,Действует,Российский рубль,7,Индонезия,1918,5000000
2,21112353,БАДАМСТЕ55*****,Страхование путешественников,10161883357,Действует,Российский рубль,10,Беларусь,1096,2000000
3,21113224,ТТЕ7227715*****,Страхование путешественников,100766461,Действует,Российский рубль,14,Грузия,1534,2000000
4,21113292,ТТЕ7227715*****,Страхование путешественников,10171604758,Действует,Российский рубль,7,Аргентина,1918,5000000


,loss_id,client_id,loss_name,loss_payout_amt
0,1,10000713193,Оказание медицинской помощи,100000
1,2,100014656,Оказание медицинской помощи,50000
2,3,10006545463,Оказание медицинской помощи,30000
3,4,10006611194,Оказание медицинской помощи,70000
4,5,1001091597,Оказание медицинской помощи,60000


## 2. Проверка структуры договоров и статусов полисов

Для итоговой витрины нужны только действующие страховые полисы.

Проверим:
- названия и типы столбцов;
- возможные пропуски;
- значения статуса договора;
- валюты, в которых представлены денежные показатели.

In [3]:
print("Столбцы contracts:")
print(contracts.columns.tolist())

print("\nТипы данных:")
print(contracts.dtypes)

print("\nПропуски:")
print(contracts.isna().sum())

print("\nСтатусы договоров:")
print(contracts["contract_status"].value_counts(dropna=False))

print("\nВалюты:")
print(contracts["currency_name"].value_counts(dropna=False))

Столбцы contracts:
['contract_id', 'contract_num', 'product_name', 'client_id', 'contract_status', 'currency_name', 'duration', 'country', 'price', 'insurance_amount']

Типы данных:
contract_id          int64
contract_num        object
product_name        object
client_id            int64
contract_status     object
currency_name       object
duration             int64
country             object
price                int64
insurance_amount     int64
dtype: object

Пропуски:
contract_id         0
contract_num        0
product_name        0
client_id           0
contract_status     0
currency_name       0
duration            0
country             0
price               0
insurance_amount    0
dtype: int64

Статусы договоров:
contract_status
Действует    3315
Завершен      396
Name: count, dtype: int64

Валюты:
currency_name
Российский рубль    3591
Доллар США           120
Name: count, dtype: int64


## 3. Отбор действующих полисов

В итоговую витрину должны попасть только действующие страховые полисы.

В таблице договоров:
- 3315 действующих договоров;
- 396 завершённых договоров.

Отфильтруем только записи со статусом `Действует`.

In [4]:
active_contracts = contracts[
    contracts["contract_status"] == "Действует"
].copy()

print("Действующих договоров:", active_contracts.shape[0])

display(active_contracts.head())

Действующих договоров: 3315


,contract_id,contract_num,product_name,client_id,contract_status,currency_name,duration,country,price,insurance_amount
0,21111219,ТТЕ7227715*****,Страхование путешественников,10161870404,Действует,Российский рубль,10,Беларусь,1096,2000000
1,21111381,ТТЕ7227715*****,Страхование путешественников,1017159879,Действует,Российский рубль,7,Индонезия,1918,5000000
2,21112353,БАДАМСТЕ55*****,Страхование путешественников,10161883357,Действует,Российский рубль,10,Беларусь,1096,2000000
3,21113224,ТТЕ7227715*****,Страхование путешественников,100766461,Действует,Российский рубль,14,Грузия,1534,2000000
4,21113292,ТТЕ7227715*****,Страхование путешественников,10171604758,Действует,Российский рубль,7,Аргентина,1918,5000000


## 4. Проверка данных клиентов и убытков

Перед объединением таблиц проверим:
- уникальность `client_id`;
- наличие дубликатов;
- пропуски;
- структуру таблицы убытков.

In [5]:
print("Уникальных client_id в clients:",
     clients["client_id"].nunique())

print("Строк в clients:",
     len(clients))

print("\nДубликаты client_id в clients:",
     clients["client_id"].duplicated().sum())

print("\nПропуски в clients:")
print(clients.isna().sum())

print("\nПропуски в losses:")
print(losses.isna().sum())

print("\nУникальных клиентов с убытками:",
     losses["client_id"].nunique())

print("Всего строк в losses:",
     len(losses))

Уникальных client_id в clients: 3711
Строк в clients: 3711

Дубликаты client_id в clients: 0

Пропуски в clients:
client_id        0
last_name        0
first_name      18
middle_name    258
age              0
sex              0
dtype: int64

Пропуски в losses:
loss_id            0
client_id          0
loss_name          0
loss_payout_amt    0
dtype: int64

Уникальных клиентов с убытками: 45
Всего строк в losses: 45


## 5. Объединение данных

Сначала объединим действующие договоры с данными клиентов по `client_id`.

Затем добавим информацию об убытках.

Так как в таблице `losses` каждый `client_id` встречается только один раз, предварительная агрегация убытков не требуется.

Для объединения используем `left join`, чтобы сохранить все действующие договоры, даже если по клиенту не было убытка.

In [6]:
data_mart = (
    active_contracts
    .merge(
        clients,
        on="client_id",
        how="left"
    )
    .merge(
        losses[["client_id", "loss_name", "loss_payout_amt"]],
        on="client_id",
        how="left"
    )
)

print("Размер витрины:", data_mart.shape)

display(data_mart.head())

Размер витрины: (3315, 17)


,contract_id,contract_num,product_name,client_id,contract_status,currency_name,duration,country,price,insurance_amount,last_name,first_name,middle_name,age,sex,loss_name,loss_payout_amt
0,21111219,ТТЕ7227715*****,Страхование путешественников,10161870404,Действует,Российский рубль,10,Беларусь,1096,2000000,А*****,Х*****,А*****,20,M,NaN,NaN
1,21111381,ТТЕ7227715*****,Страхование путешественников,1017159879,Действует,Российский рубль,7,Индонезия,1918,5000000,З*****,Х*****,М*****,55,M,NaN,NaN
2,21112353,БАДАМСТЕ55*****,Страхование путешественников,10161883357,Действует,Российский рубль,10,Беларусь,1096,2000000,Б*****,В*****,Щ*****,61,F,NaN,NaN
3,21113224,ТТЕ7227715*****,Страхование путешественников,100766461,Действует,Российский рубль,14,Грузия,1534,2000000,И*****,У*****,Н*****,25,M,NaN,NaN
4,21113292,ТТЕ7227715*****,Страхование путешественников,10171604758,Действует,Российский рубль,7,Аргентина,1918,5000000,З*****,Ч*****,Г*****,33,M,NaN,NaN


## 6. Проверка результата объединения

После объединения количество строк должно совпадать с количеством действующих договоров.

Также проверим уникальность `contract_id` и количество договоров, для которых найдена информация об убытках.

In [7]:
print("Строк в витрине:", len(data_mart))
print("Уникальных contract_id:", data_mart["contract_id"].nunique())

print(
    "Договоров с информацией об убытке:",
    data_mart["loss_payout_amt"].notna().sum()
)

print("\nПропуски после объединения:")
print(data_mart.isna().sum())

Строк в витрине: 3315
Уникальных contract_id: 3315
Договоров с информацией об убытке: 45

Пропуски после объединения:
contract_id            0
contract_num           0
product_name           0
client_id              0
contract_status        0
currency_name          0
duration               0
country                0
price                  0
insurance_amount       0
last_name              0
first_name            17
middle_name          226
age                    0
sex                    0
loss_name           3270
loss_payout_amt     3270
dtype: int64


## 7. Формирование инициалов страхователя

По условию необходимо добавить инициалы страхователя в формате `Ф. И. О.`.

Фамилия заполнена у всех клиентов, однако в имени и отчестве встречаются пропуски. Поэтому будем формировать инициалы только из имеющихся значений, не создавая букв из пропущенных данных.

In [8]:
def make_initials(row):
    initials = []

    for column in ["last_name", "first_name", "middle_name"]:
        value = row[column]

        if pd.notna(value) and not str(value).startswith("-"):
            initials.append(str(value)[0] + ".")

    return " ".join(initials)

data_mart["initials"] = data_mart.apply(make_initials, axis=1)

display(
    data_mart[
        ["last_name", "first_name", "middle_name", "initials"]
    ].head(10)
)

,last_name,first_name,middle_name,initials
0,А*****,Х*****,А*****,А. Х. А.
1,З*****,Х*****,М*****,З. Х. М.
2,Б*****,В*****,Щ*****,Б. В. Щ.
3,И*****,У*****,Н*****,И. У. Н.
4,З*****,Ч*****,Г*****,З. Ч. Г.
5,Д*****,В*****,-*****,Д. В.
6,Д*****,З*****,Г*****,Д. З. Г.
7,Д*****,М*****,Д*****,Д. М. Д.
8,Д*****,Р*****,В*****,Д. Р. В.
9,З*****,Ш*****,Ю*****,З. Ш. Ю.


## 8. Проверка формирования инициалов

Проверим строки, где отсутствует имя или отчество, чтобы убедиться, что инициалы формируются корректно и пропущенные значения не превращаются в текст `nan`.

In [9]:
display(
    data_mart[
        data_mart["first_name"].isna()
        | data_mart["middle_name"].isna()
    ][
        ["last_name", "first_name", "middle_name", "initials"]
    ].head(15)
)

,last_name,first_name,middle_name,initials
35,F*****,T*****,NaN,F. T.
39,Ё*****,Г*****,NaN,Ё. Г.
41,К*****,С*****,NaN,К. С.
48,Б*****,NaN,NaN,Б.
50,Y*****,K*****,NaN,Y. K.
63,A*****,E*****,NaN,A. E.
100,Г*****,П*****,NaN,Г. П.
160,C*****,H*****,NaN,C. H.
182,И*****,Г*****,NaN,И. Г.
195,З*****,В*****,NaN,З. В.


## 9. Получение актуального курса доллара через API ЦБ РФ

Для пересчёта денежных показателей в доллары США получим курс USD через API Центрального банка РФ.

Согласно условию задачи, под актуальным курсом будем понимать курс ЦБ за вчерашний день.

API ЦБ возвращает данные в формате XML, поэтому после получения ответа найдём валюту с кодом `USD` и извлечём её курс.

In [10]:
from datetime import datetime, timedelta
import xml.etree.ElementTree as ET

yesterday = datetime.now() - timedelta(days=1)
date_req = yesterday.strftime("%d/%m/%Y")

url = "https://www.cbr.ru/scripts/XML_daily.asp"

response = requests.get(
    url,
    params={"date_req": date_req}
)

print("HTTP status:", response.status_code)
print("Запрошенная дата:", date_req)

HTTP status: 200
Запрошенная дата: 22/09/2026


## 10. Извлечение курса доллара из XML

Ответ API ЦБ получен успешно.

Теперь разберём XML-ответ, найдём валюту с кодом `USD` и извлечём:
- номинал;
- значение курса.

Так как курс ЦБ может быть указан не за 1 единицу валюты, учтём `Nominal` при расчёте.

In [11]:
root = ET.fromstring(response.content)

usd_rate = None

for valute in root.findall("Valute"):
    char_code = valute.find("CharCode").text

    if char_code == "USD":
        nominal = int(valute.find("Nominal").text)
        value = float(
            valute.find("Value").text.replace(",", ".")
        )

        usd_rate = value / nominal
        break

print("Курс USD:", usd_rate, "руб. за 1 USD")

Курс USD: 84.0954 руб. за 1 USD


## 11. Проверка валюты договоров с убытками

В таблице `losses` валюта выплаты отдельно не указана.

Поэтому перед пересчётом `loss_payout_amt` проверим, в какой валюте оформлены действующие договоры клиентов, у которых зафиксирован убыток.

Это позволит избежать необоснованного предположения о валюте выплаты.

In [12]:
loss_currency_check = data_mart[
    data_mart["loss_payout_amt"].notna()
][
    ["client_id", "currency_name", "loss_payout_amt"]
]

print("Валюты договоров клиентов с убытками:")
print(loss_currency_check["currency_name"].value_counts())

display(loss_currency_check.head(10))

Валюты договоров клиентов с убытками:
currency_name
Российский рубль    43
Доллар США           2
Name: count, dtype: int64


,client_id,currency_name,loss_payout_amt
16,10000713193,Российский рубль,100000.0
32,100014656,Российский рубль,50000.0
90,1000579095,Доллар США,20000.0
100,10006545463,Российский рубль,30000.0
109,10006611194,Российский рубль,70000.0
245,1001091597,Российский рубль,60000.0
289,1001198959,Российский рубль,50000.0
456,100193151,Российский рубль,100000.0
549,10034571331,Российский рубль,50000.0
646,10041665730,Российский рубль,20000.0


## 12. Пересчёт денежных показателей в доллары США

В таблице убытков отдельная валюта выплаты не указана.

Поэтому в рамках работы принимаем допущение, что сумма выплаты по убытку указана в той же валюте, что и соответствующий страховой договор.

Для договоров в российских рублях денежные показатели переводим в доллары США по полученному курсу ЦБ:

`сумма в USD = сумма в RUB / курс USD`

Если договор изначально указан в долларах США, значение оставляем без изменения.

Пересчитаем:
- стоимость полиса `price`;
- страховую сумму `insurance_amount`;
- сумму выплаты по убытку `loss_payout_amt`.

In [13]:
def convert_to_usd(amount, currency):
    if pd.isna(amount):
        return None

    if currency == "Российский рубль":
        return amount / usd_rate

    if currency == "Доллар США":
        return amount

    return None

data_mart["price_usd"] = data_mart.apply(
    lambda row: convert_to_usd(
        row["price"],
        row["currency_name"]
    ),
    axis=1
)

data_mart["insurance_amount_usd"] = data_mart.apply(
    lambda row: convert_to_usd(
        row["insurance_amount"],
        row["currency_name"]
    ),
    axis=1
)

data_mart["loss_payout_amt_usd"] = data_mart.apply(
    lambda row: convert_to_usd(
        row["loss_payout_amt"],
        row["currency_name"]
    ),
    axis=1
)

## 13. Проверка пересчёта валют

Проверим отдельно рублёвые и долларовые договоры, чтобы убедиться, что:
- рублёвые суммы пересчитаны по курсу;
- долларовые суммы остались без изменения.

In [14]:
display(
    data_mart[
        [
            "currency_name",
            "price",
            "price_usd",
            "insurance_amount",
            "insurance_amount_usd",
            "loss_payout_amt",
            "loss_payout_amt_usd"
        ]
    ].groupby("currency_name").head(3)
)

,currency_name,price,price_usd,insurance_amount,insurance_amount_usd,loss_payout_amt,loss_payout_amt_usd
0,Российский рубль,1096,13.032817,2000000,23782.513669,NaN,NaN
1,Российский рубль,1918,22.807431,5000000,59456.284172,NaN,NaN
2,Российский рубль,1096,13.032817,2000000,23782.513669,NaN,NaN
56,Доллар США,66,66.000000,100000,100000.000000,NaN,NaN
81,Доллар США,55,55.000000,100000,100000.000000,NaN,NaN
90,Доллар США,77,77.000000,100000,100000.000000,20000.0,20000.0


## 14. Формирование итоговой витрины

Сформируем итоговый набор столбцов.

В витрине оставим:
- информацию о страховом договоре;
- инициалы клиента;
- пол и возраст;
- информацию об убытке;
- исходные денежные показатели;
- соответствующие значения в долларах США.

In [15]:
final_mart = data_mart[
    [
        "contract_id",
        "contract_num",
        "product_name",
        "client_id",
        "initials",
        "sex",
        "age",
        "contract_status",
        "currency_name",
        "duration",
        "country",
        "price",
        "price_usd",
        "insurance_amount",
        "insurance_amount_usd",
        "loss_name",
        "loss_payout_amt",
        "loss_payout_amt_usd"
    ]
].copy()

display(final_mart.head())

,contract_id,contract_num,product_name,client_id,initials,sex,age,contract_status,currency_name,duration,country,price,price_usd,insurance_amount,insurance_amount_usd,loss_name,loss_payout_amt,loss_payout_amt_usd
0,21111219,ТТЕ7227715*****,Страхование путешественников,10161870404,А. Х. А.,M,20,Действует,Российский рубль,10,Беларусь,1096,13.032817,2000000,23782.513669,NaN,NaN,NaN
1,21111381,ТТЕ7227715*****,Страхование путешественников,1017159879,З. Х. М.,M,55,Действует,Российский рубль,7,Индонезия,1918,22.807431,5000000,59456.284172,NaN,NaN,NaN
2,21112353,БАДАМСТЕ55*****,Страхование путешественников,10161883357,Б. В. Щ.,F,61,Действует,Российский рубль,10,Беларусь,1096,13.032817,2000000,23782.513669,NaN,NaN,NaN
3,21113224,ТТЕ7227715*****,Страхование путешественников,100766461,И. У. Н.,M,25,Действует,Российский рубль,14,Грузия,1534,18.241188,2000000,23782.513669,NaN,NaN,NaN
4,21113292,ТТЕ7227715*****,Страхование путешественников,10171604758,З. Ч. Г.,M,33,Действует,Российский рубль,7,Аргентина,1918,22.807431,5000000,59456.284172,NaN,NaN,NaN


## 15. Финальная проверка витрины

Перед сохранением проверим:
- количество строк;
- уникальность договоров;
- наличие только действующих полисов;
- количество пропусков;
- наличие рассчитанных долларовых показателей.

In [16]:
print("Размер итоговой витрины:", final_mart.shape)

print(
    "Уникальных contract_id:",
    final_mart["contract_id"].nunique()
)

print("\nСтатусы договоров:")
print(final_mart["contract_status"].value_counts())

print("\nПропуски:")
print(final_mart.isna().sum())

print("\nВалюты:")
print(final_mart["currency_name"].value_counts())

Размер итоговой витрины: (3315, 18)
Уникальных contract_id: 3315

Статусы договоров:
contract_status
Действует    3315
Name: count, dtype: int64

Пропуски:
contract_id                0
contract_num               0
product_name               0
client_id                  0
initials                   0
sex                        0
age                        0
contract_status            0
currency_name              0
duration                   0
country                    0
price                      0
price_usd                  0
insurance_amount           0
insurance_amount_usd       0
loss_name               3270
loss_payout_amt         3270
loss_payout_amt_usd     3270
dtype: int64

Валюты:
currency_name
Российский рубль    3210
Доллар США           105
Name: count, dtype: int64


## 16. Округление денежных показателей

Для удобства представления итоговых данных округлим рассчитанные суммы в долларах США до двух знаков после запятой.

In [17]:
usd_columns = [
    "price_usd",
    "insurance_amount_usd",
    "loss_payout_amt_usd"
]

final_mart[usd_columns] = final_mart[usd_columns].round(2)

display(
    final_mart[
        [
            "currency_name",
            "price",
            "price_usd",
            "insurance_amount",
            "insurance_amount_usd",
            "loss_payout_amt",
            "loss_payout_amt_usd"
        ]
    ].head(10)
)

,currency_name,price,price_usd,insurance_amount,insurance_amount_usd,loss_payout_amt,loss_payout_amt_usd
0,Российский рубль,1096,13.03,2000000,23782.51,NaN,NaN
1,Российский рубль,1918,22.81,5000000,59456.28,NaN,NaN
2,Российский рубль,1096,13.03,2000000,23782.51,NaN,NaN
3,Российский рубль,1534,18.24,2000000,23782.51,NaN,NaN
4,Российский рубль,1918,22.81,5000000,59456.28,NaN,NaN
5,Российский рубль,4932,58.65,3000000,35673.77,NaN,NaN
6,Российский рубль,4932,58.65,3000000,35673.77,NaN,NaN
7,Российский рубль,1918,22.81,5000000,59456.28,NaN,NaN
8,Российский рубль,4932,58.65,3000000,35673.77,NaN,NaN
9,Российский рубль,1918,22.81,5000000,59456.28,NaN,NaN


## 17. Сохранение итоговой витрины

Сохраним полученную витрину в Excel-файл.

В итоговом файле содержатся только действующие страховые полисы, дополненные данными клиентов, информацией об убытках и денежными показателями в долларах США.

In [18]:
output_file = "insurance_data_mart.xlsx"

final_mart.to_excel(
    output_file,
    index=False
)

print("Файл сохранен:", output_file)

Файл сохранен: insurance_data_mart.xlsx


## 18. Итог

В результате работы сформирована единая витрина данных по действующим страховым полисам.

В ходе работы:

- загружены данные из трёх Excel-файлов;
- отобраны только действующие договоры;
- данные договоров объединены с данными клиентов и убытками;
- сформированы инициалы страхователей с учётом пропущенных имени или отчества;
- через API Центрального банка РФ получен актуальный курс доллара США;
- стоимость полиса, страховая сумма и сумма выплаты по убытку пересчитаны в доллары США;
- для исходно долларовых договоров денежные значения оставлены без изменения;
- сформирована итоговая витрина из 3315 действующих договоров;
- результат сохранён в Excel-файл `insurance_data_mart.xlsx`.

При пересчёте выплаты по убытку принято допущение, что её валюта совпадает с валютой соответствующего страхового договора, так как отдельная валюта выплаты в исходной таблице `losses` не указана.